# Скорость LazyFrame и DataFrame

Импорт библиотек

In [1]:
import polars as pl
import warnings
warnings.filterwarnings("ignore")

Лучший способ сравнить быстродействие *LazyFrame* и *DataFrame* - применить их к одним и тем же данным и одинаковым операциям.
Проведем два теста:
1. Датасет на основе csv документа, в котором 7 216 957 строк.
2. Датасет на основе parquet документа, в котором 34 646 258 строк.

### Испытание 1.

Посмотрим на данные

In [2]:
csv_doc =  "btcusd_1-min_data.csv"

In [3]:
df = pl.read_csv(source=csv_doc)
lf = pl.scan_csv(source=csv_doc)

print(df.head(4))
print(lf)


shape: (4, 6)
┌───────────┬──────┬──────┬──────┬───────┬────────┐
│ Timestamp ┆ Open ┆ High ┆ Low  ┆ Close ┆ Volume │
│ ---       ┆ ---  ┆ ---  ┆ ---  ┆ ---   ┆ ---    │
│ f64       ┆ f64  ┆ f64  ┆ f64  ┆ f64   ┆ f64    │
╞═══════════╪══════╪══════╪══════╪═══════╪════════╡
│ 1.3254e9  ┆ 4.58 ┆ 4.58 ┆ 4.58 ┆ 4.58  ┆ 0.0    │
│ 1.3254e9  ┆ 4.58 ┆ 4.58 ┆ 4.58 ┆ 4.58  ┆ 0.0    │
│ 1.3254e9  ┆ 4.58 ┆ 4.58 ┆ 4.58 ┆ 4.58  ┆ 0.0    │
│ 1.3254e9  ┆ 4.58 ┆ 4.58 ┆ 4.58 ┆ 4.58  ┆ 0.0    │
└───────────┴──────┴──────┴──────┴───────┴────────┘
naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Csv SCAN [btcusd_1-min_data.csv]
PROJECT */6 COLUMNS


Данные представляют собой временной ряд,связанный с финансовыми рынками.

Описание колонок:
- `Timestamp`: Это временная метка, представляющая момент времени, когда были собраны данные. Значение в формате f64 указывает на количество миллисекунд с начала эпохи Unix (1 января 1970 года).
- `Open`: Цена открытия за определенный период времени. Это цена, по которой актив начал торговаться в начале этого периода.
- `High`: Максимальная цена, достигнутая активом в течение данного периода.
- `Low`: Минимальная цена, достигнутая активом в течение данного периода.
- `Close`: Цена закрытия за определенный период времени. Это цена, по которой актив завершил торги в конце этого периода.
- `Volume`: Объем торгов, который показывает количество акций или контрактов, которые были куплены и проданы в течение данного периода. Значение 0.0 может указывать на отсутствие торгов за этот период.

Посмотрим, за какое время *polars DataFrame* выполнит чтение документа, преобразование, фильтрацию, группировку, агрегации и сортировку.

In [4]:
%%time

# Загружаем DataFrame
df = pl.read_csv(source=csv_doc)

# Делаем преобразования
df\
    .with_columns(
        pl.col("Timestamp").cast(pl.Datetime).dt.with_time_unit("ms") # Приводим поле к правильному формату данных
    )\
    .filter(
        (pl.col("Timestamp") > pl.lit("1970-01-20").str.to_datetime()),
        (pl.col("Volume") > 1)
    )\
    .group_by(
        pl.col("Timestamp").dt.date().alias("date") # Группируем по дате
    )\
    .agg(
        pl.col("Open").first().alias("Open"),       # Первое значение
        pl.col("High").max().alias("High"),         # Максимум
        pl.col("Low").min().alias("Low"),           # Минимум
        pl.col("Close").last().alias("Close"),      # Последнее значение
        pl.col("Volume").sum().alias("Volume")      # Сумма объёмов
    )\
    .sort("date")  # Сортируем по дате для порядка


CPU times: total: 2.94 s
Wall time: 457 ms


date,Open,High,Low,Close,Volume
date,f64,f64,f64,f64,f64
1970-01-20,41666.27,73794.0,15479.0,60750.0,1.8766e6
1970-01-21,60699.0,124517.0,58867.0,115319.0,529659.862458


Теперь проделаем тоже самое с *LazyFrame*.

In [5]:
%%time

# Сканируем данные
lf = pl.scan_csv(source=csv_doc)

# Делаем преобразования
lf\
    .with_columns(
        pl.col("Timestamp").cast(pl.Datetime).dt.with_time_unit("ms") # Приводим поле к правильному формату данных
    )\
    .filter(
        (pl.col("Timestamp") > pl.lit("1970-01-20").str.to_datetime()),
        (pl.col("Volume") > 1)
    )\
    .group_by(
        pl.col("Timestamp").dt.date().alias("date") # Группируем по дате
    )\
    .agg(
        pl.col("Open").first().alias("Open"),       # Первое значение
        pl.col("High").max().alias("High"),         # Максимум
        pl.col("Low").min().alias("Low"),           # Минимум
        pl.col("Close").last().alias("Close"),      # Последнее значение
        pl.col("Volume").sum().alias("Volume")      # Сумма объёмов
    )\
    .sort("date").collect()


CPU times: total: 2.86 s
Wall time: 452 ms


date,Open,High,Low,Close,Volume
date,f64,f64,f64,f64,f64
1970-01-20,41666.27,73794.0,15479.0,60750.0,1.8766e6
1970-01-21,60699.0,124517.0,58867.0,115319.0,529659.862458


Несмотря на то, что *LazyFrame* предназначен для оптимизации производительности за счёт отложенного выполнения и оптимизации запросов, в данном случае он не дал выигрыша в скорости, а даже показал незначительно большее время выполнения (418 миллисекунд против 437 миллисекунд).

Это связано с не очень большим объёмом данных: датасет содержит всего 7 216 957 строк, и классический *polars DataFrame* чувствует себя хорошо.

Оптимизатор *LazyFrame* не получает преимущества при малых объёмах, потому что накладные расходы на построение и оптимизацию DAG (Directed Acyclic Graph) запроса могут быть сопоставимы или даже превышать выгоду.

Теперь разберём датасет чуть-чуть побольше.

### Испытание 2.

Будем читать данные из "паркета":

In [6]:
dfp = pl.read_parquet("all_stock_data.parquet")
lfp = pl.scan_parquet("all_stock_data.parquet")

dfp

Date,Ticker,Open,High,Low,Close,Volume,Dividends,Stock Splits
date,str,f64,f64,f64,f64,f64,f64,f64
1962-01-02,"""ED""",0.0,0.265828,0.261788,0.261788,25600.0,0.0,0.0
1962-01-02,"""CVX""",0.0,0.046809,0.046069,0.046809,105840.0,0.0,0.0
1962-01-02,"""GD""",0.0,0.210033,0.203061,0.20829,2.648e6,0.0,0.0
1962-01-02,"""BP""",0.0,0.141439,0.139528,0.139528,77440.0,0.0,0.0
1962-01-02,"""MSI""",0.0,0.764923,0.745254,0.75181,65671.0,0.0,0.0
…,…,…,…,…,…,…,…,…
2024-11-04,"""NEOG""",14.49,14.58,14.34,14.345,18972.0,0.0,0.0
2024-11-04,"""ENLV""",1.4,1.4289,1.33,1.35,28794.0,0.0,0.0
2024-11-04,"""FAMI""",0.32,0.32,0.3001,0.3001,77650.0,0.0,0.0


[Датасет](https://www.kaggle.com/datasets/jakewright/9000-tickers-of-stock-market-data-full-history?select=all_stock_data.parquet) представляет собой данные фондового рынка: более 9000 [тикеров](https://ru.wikipedia.org/wiki/%D0%A2%D0%B8%D0%BA%D0%B5%D1%80) (с 1962 года по настоящее время). Он включает в себя важную информацию о ежедневных торгах, что делает его подходящим для различных видов финансового анализа, изучения тенденций и разработки алгоритмических торговых моделей.

Столбцы:
- `Date`: дата записи торговых данных.
- `Тикер`: биржевой код компании.
- `Open`: цена открытия торгов акциями в день.
- `High`: максимальная цена, достигнутая в течение торгового дня.
- `Low`: минимальная цена, достигнутая в течение торгового дня.
- `Close`: цена закрытия торгов акциями в этот день.
- `Volume`: общее количество акций, проданных в течение дня.
- `Dividends`: денежные дивиденды, выплаченные на дату, если применимо.
- `Stock Splits`: коэффициент дробления акций на указанную дату, если имело место дробление.

Приступим к тесту. Найдём топ-10 тикеров (по средней цене закрытия), которые торговались с 1990 года, имели объём > 1000, присутствовали на рынке после 2020 года и имели более 100 торговых дней. Запустим сначала *LazyFrame*.

In [13]:
%%time

lfp = pl.scan_parquet("all_stock_data.parquet")

result_lf = (
    lfp
    .filter(
        (pl.col("Date") >= pl.date(1990, 1, 1)) &
        (pl.col("Volume") > 1000)
    )
    .filter(
        pl.col("Ticker").is_in(
            lfp.filter(pl.col("Date") >= pl.date(2020, 1, 1))
              .select("Ticker")
              .unique()
              .collect()  # ← подзапрос: тикеры после 2020
        )
    )
    .group_by("Ticker")
    .agg(
        pl.col("Close").mean().alias("avg_close"),
        pl.len().alias("trade_days")
    )
    .filter(pl.col("trade_days") > 100)  # только активные тикеры
    .sort("avg_close", descending=True)
    .limit(10)  # топ-10
    .collect()
)

print("LazyFrame результат:")
print(result_lf)

LazyFrame результат:
shape: (10, 3)
┌────────┬───────────────┬────────────┐
│ Ticker ┆ avg_close     ┆ trade_days │
│ ---    ┆ ---           ┆ ---        │
│ str    ┆ f64           ┆ u32        │
╞════════╪═══════════════╪════════════╡
│ PTPIF  ┆ 2.2582e26     ┆ 140        │
│ PPERF  ┆ 1.6371e8      ┆ 1275       │
│ NDEKF  ┆ 285658.282098 ┆ 592        │
│ BRK-A  ┆ 127175.894059 ┆ 6788       │
│ DNZOF  ┆ 50588.876299  ┆ 1204       │
│ UVXY   ┆ 30432.181544  ┆ 1985       │
│ ADTX   ┆ 5773.984989   ┆ 262        │
│ FFIE   ┆ 4567.484561   ┆ 587        │
│ GPUS   ┆ 3710.003426   ┆ 666        │
│ PIXY   ┆ 3608.418304   ┆ 295        │
└────────┴───────────────┴────────────┘
CPU times: total: 8.53 s
Wall time: 1.73 s


Отработал за 1,73 секунды. Теперь посмотрим на *DataFrame*.

In [12]:
%%time

dfp = pl.read_parquet("all_stock_data.parquet")

# Сначала получаем список тикеров после 2020
tickers_after_2020 = dfp.filter(pl.col("Date") >= pl.date(2020, 1, 1))["Ticker"].unique()

# Потом фильтруем основной датафрейм
result_df = (
    dfp
    .filter(
        (pl.col("Date") >= pl.date(1990, 1, 1)) &
        (pl.col("Volume") > 1000) &
        (pl.col("Ticker").is_in(tickers_after_2020))
    )
    .group_by("Ticker")
    .agg(
        pl.col("Close").mean().alias("avg_close"),
        pl.len().alias("trade_days")
    )
    .filter(pl.col("trade_days") > 100)
    .sort("avg_close", descending=True)
    .limit(10)
)

print("DataFrame результат:")
print(result_df)

DataFrame результат:
shape: (10, 3)
┌────────┬───────────────┬────────────┐
│ Ticker ┆ avg_close     ┆ trade_days │
│ ---    ┆ ---           ┆ ---        │
│ str    ┆ f64           ┆ u32        │
╞════════╪═══════════════╪════════════╡
│ PTPIF  ┆ 2.2582e26     ┆ 140        │
│ PPERF  ┆ 1.6371e8      ┆ 1275       │
│ NDEKF  ┆ 285658.282098 ┆ 592        │
│ BRK-A  ┆ 127175.894059 ┆ 6788       │
│ DNZOF  ┆ 50588.876299  ┆ 1204       │
│ UVXY   ┆ 30432.181544  ┆ 1985       │
│ ADTX   ┆ 5773.984989   ┆ 262        │
│ FFIE   ┆ 4567.484561   ┆ 587        │
│ GPUS   ┆ 3710.003426   ┆ 666        │
│ PIXY   ┆ 3608.418304   ┆ 295        │
└────────┴───────────────┴────────────┘
CPU times: total: 12.8 s
Wall time: 3.99 s


Сравнивая результаты:
 - LazyFrame: Wall time = 1.73 с;
 - DataFrame: Wall time = 3.99 с.

Таким образом, *LazyFrame* выполнился примерно в 2.3 раза быстрее. Это связано с тем, что оптимизатор *polars* смог эффективно перестроить запрос:
- отложил чтение данных до самого конца,
- применил предикат-пушдаун (predicate pushdown) для фильтрации прямо на этапе чтения Parquet-файла,
- избежал промежуточного материализованного датафрейма при выполнении подзапроса (несмотря на `.collect()` внутри фильтра, общий план всё равно частично оптимизировался).
  
В то же время подход с обычным *DataFrame* потребовал:
- полной загрузки всего Parquet-файла в память (`pl.read_parquet`),
- отдельного прохода для получения списка тикеров после 2020 года,
- затем ещё одного полного прохода по уже загруженному фрейму данных для основной фильтрации и агрегации.

Хоть объём данных (~7.2 млн строк) не является гигантским, *LazyFrame* продемонстрировал существенное преимущество благодаря более эффективному управлению потоком данных и минимизации лишних операций в памяти.

Таким образом, даже на средних по размеру датасетах Lazy API от Polars может давать заметный прирост производительности - особенно при сложных многоэтапных запросах с фильтрацией и подзапросами.

### Общий вывод

Когда использовать *DataFrame* (Eager), а когда *LazyFrame* (Lazy)?

Выбор между ними зависит от размера данных, сложности запроса и ограничений по памяти.

Использовать *LazyFrame* рекомендуется в большинстве случаев, особенно когда:
1. Большие данные (десятки миллионов строк и выше).
2. Выполняются сложные трансформации: фильтрация, группировка, джойны, подзапросы, сортировка.
3. Важна производительность и экономия памяти.
4. Работа с форматами, поддерживающими чтение по частям (Parquet, CSV с scan_*).

Использовать *DataFrame* уместно, когда:
1. Небольшие данные (десятки или сотни тысяч строк).
2. Нужен интерактивный анализ/отладка.
3. Требуется интеграция с библиотеками, ожидающими pandas-like объект (например, matplotlib, scikit-learn).
4. Выполняются простые операции без цепочек трансформаций.

*LazyFrame* - это мощный инструмент для эффективной обработки больших данных, а *DataFrame* - удобный инструмент для быстрой работы с небольшими наборами.
Если ограничены в ресурсах, то правильный выбор режима позволяет достичь максимальной скорости и минимального использования ресурсов. 